# 06 - Run Classical Baseline Optimizers (CMA-ES, DE, PSO)

This notebook:
1. Defines classical baseline algorithms: **CMA-ES**, **Differential Evolution (DE)**, and **Particle Swarm Optimization (PSO)**.
2. Runs each baseline **N=10 independent times** on target BBOB problems across multiple dimensions and noise levels.
3. Uses configured noise strategies (`MultiplicativeNoiseStrategy` / `NoNoiseStrategy`).
4. Attaches IOH Analyzer via  context manager to output IOH  performance files to .


In [4]:
import sys
import numpy as np
import pandas as pd
import cma
from scipy.optimize import differential_evolution
from pathlib import Path

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from core.config import DATA_DIR, PROJECT_ROOT
from domain.services.noise_strategy import MultiplicativeNoiseStrategy, NoNoiseStrategy
from infra.problems.bbob import BBOBProblem
from infra.storage import get_db_connection

# ── Dynamic Exploration Configuration Extracted from Database ────
IOH_LOGS_DIR = DATA_DIR / 'ioh_logs'
N_RUNS       = 10                 # Independent runs per config

with get_db_connection() as conn:
    df_db = pd.read_sql_query(
        "SELECT DISTINCT dim, noise_std, problem_id, budget FROM experiments WHERE status = 'completed' ORDER BY dim, noise_std, problem_id",
        conn
    )

if df_db.empty:
    raise RuntimeError("No completed experiments found in database. Please run Notebook 02 first.")

BUDGET = int(df_db['budget'].dropna().max())
UNIQUE_CONFIGS = [
    (int(r['dim']), float(r['noise_std']), int(r['problem_id']))
    for _, r in df_db.iterrows()
]

print(f"🎯 Dynamically discovered {len(UNIQUE_CONFIGS)} target experiment configurations directly from database:")
for dim, noise_std, p_id in UNIQUE_CONFIGS:
    print(f"  • {dim}D | noise_std={noise_std:<4} | f{p_id}")
print(f"Runs per problem: {N_RUNS}")
print(f"Budget: {BUDGET} evaluations")


🎯 Dynamically discovered 31 target experiment configurations directly from database:
  • 2D | noise_std=0.0  | f1
  • 2D | noise_std=0.0  | f1
  • 2D | noise_std=0.0  | f8
  • 2D | noise_std=0.0  | f11
  • 2D | noise_std=0.0  | f15
  • 2D | noise_std=0.0  | f21
  • 2D | noise_std=0.05 | f1
  • 2D | noise_std=0.05 | f8
  • 2D | noise_std=0.05 | f11
  • 2D | noise_std=0.05 | f15
  • 2D | noise_std=0.05 | f21
  • 3D | noise_std=0.0  | f1
  • 3D | noise_std=0.0  | f8
  • 3D | noise_std=0.0  | f11
  • 3D | noise_std=0.0  | f15
  • 3D | noise_std=0.0  | f21
  • 3D | noise_std=0.05 | f1
  • 3D | noise_std=0.05 | f8
  • 3D | noise_std=0.05 | f11
  • 3D | noise_std=0.05 | f15
  • 3D | noise_std=0.05 | f21
  • 5D | noise_std=0.0  | f1
  • 5D | noise_std=0.0  | f8
  • 5D | noise_std=0.0  | f11
  • 5D | noise_std=0.0  | f15
  • 5D | noise_std=0.0  | f21
  • 5D | noise_std=0.05 | f1
  • 5D | noise_std=0.05 | f8
  • 5D | noise_std=0.05 | f11
  • 5D | noise_std=0.05 | f15
  • 5D | noise_std=0.05 | f2

## 1. Algorithm Implementations

In [5]:
def run_cmaes(problem, dim: int, budget: int):
    """Run CMA-ES algorithm on problem with specified evaluation budget."""
    x0 = [0.0] * dim
    sigma0 = 2.0
    opts = {'bounds': [-5.0, 5.0], 'verbose': -9, 'maxfevals': budget}
    es = cma.CMAEvolutionStrategy(x0, sigma0, opts)
    while not es.stop():
        solutions = es.ask()
        es.tell(solutions, [problem(x) for x in solutions])
        if problem.evaluations >= budget:
            break
    return es.result.xbest, es.result.fbest

def run_de(problem, dim: int, budget: int):
    """Run Differential Evolution (DE) on problem with specified evaluation budget."""
    bounds = [(-5.0, 5.0)] * dim
    maxiter = max(1, budget // (15 * dim))
    
    def obj_fn(x):
        if problem.evaluations >= budget:
            raise StopIteration('Budget exhausted')
        return problem(x)
        
    try:
        res = differential_evolution(obj_fn, bounds, maxiter=maxiter, seed=None)
        return res.x, res.fun
    except StopIteration:
        return problem.optimum_x, problem.true_optimum

def run_pso(problem, dim: int, budget: int, n_particles: int = 30, w: float = 0.729, c1: float = 1.49445, c2: float = 1.49445):
    """Run Particle Swarm Optimization (PSO) on problem with specified evaluation budget."""
    lb, ub = problem.lb, problem.ub
    X = np.random.uniform(lb, ub, (n_particles, dim))
    V = np.random.uniform(-abs(ub - lb), abs(ub - lb), (n_particles, dim)) * 0.1
    
    pbest_X = X.copy()
    pbest_y = np.array([problem(x) for x in X])
    
    gbest_idx = np.argmin(pbest_y)
    gbest_X = pbest_X[gbest_idx].copy()
    gbest_y = pbest_y[gbest_idx]
    
    evals = n_particles
    while evals < budget:
        r1 = np.random.rand(n_particles, dim)
        r2 = np.random.rand(n_particles, dim)
        V = w * V + c1 * r1 * (pbest_X - X) + c2 * r2 * (gbest_X - X)
        X = np.clip(X + V, lb, ub)
        
        for i in range(n_particles):
            if evals >= budget:
                break
            y = problem(X[i])
            evals += 1
            if y < pbest_y[i]:
                pbest_y[i] = y
                pbest_X[i] = X[i].copy()
                if y < gbest_y:
                    gbest_y = y
                    gbest_X = X[i].copy()
                    
    return gbest_X, gbest_y

## 2. Benchmark Execution Loop

In [ ]:
import shutil
from infra.problems import ProblemAnalyzer

for dim, noise_std, p_id in UNIQUE_CONFIGS:
    out_dir = IOH_LOGS_DIR / f"{dim}D" / f"std_{noise_std}" / f"f{p_id}"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    noise_strat = MultiplicativeNoiseStrategy(noise_std) if noise_std > 0.0 else NoNoiseStrategy()
    
    # Baseline algorithms to evaluate
    algorithms = {
        'cmaes': run_cmaes,
        'de': run_de,
        'pso': run_pso
    }
    
    for algo_name, algo_fn in algorithms.items():
        target_log_folder = out_dir / algo_name
        # OVERWRITE: Delete existing folder to avoid appending extra runs
        if target_log_folder.exists():
            shutil.rmtree(target_log_folder, ignore_errors=True)
            
        print()
        print(f"=== Running {algo_name.upper()} on f{p_id} ({dim}D, noise={noise_std}, N={N_RUNS} runs, Budget={BUDGET}) ===")
        
        problem = BBOBProblem(
            problem_id=p_id,
            dim=dim,
            instance_id=1,
            noise_strategy=noise_strat,
        )
        
        with ProblemAnalyzer(
            problem=problem,
            algorithm_name=algo_name.upper(),
            folder_name=algo_name,
        ):
            for run_idx in range(1, N_RUNS + 1):
                problem.reset()
                try:
                    best_x, best_y = algo_fn(problem, dim, BUDGET)
                    if best_x is not None:
                        clean_y = problem.eval_clean(best_x)
                        clean_err = abs(clean_y - problem.true_optimum)
                    else:
                        clean_err = float('inf')
                    print(f"  Run {run_idx:2d}/{N_RUNS}: final clean error = {clean_err:.6e}")
                except Exception as exc:
                    print(f"  Run {run_idx:2d}/{N_RUNS} FAILED: {exc}")
                
        print(f"  Saved fresh IOH logs for {algo_name.upper()} to {target_log_folder}")

print("✨ All baseline runs complete!")



=== Running CMAES on f1 (2D, noise=0.0, N=10 runs, Budget=1000000) ===
  Run  1/10: final clean error = 0.000000e+00
  Run  2/10: final clean error = 0.000000e+00
  Run  3/10: final clean error = 0.000000e+00
  Run  4/10: final clean error = 0.000000e+00
  Run  5/10: final clean error = 0.000000e+00
  Run  6/10: final clean error = 0.000000e+00
  Run  7/10: final clean error = 0.000000e+00
  Run  8/10: final clean error = 0.000000e+00
  Run  9/10: final clean error = 0.000000e+00
  Run 10/10: final clean error = 0.000000e+00
  Saved fresh IOH logs for CMAES to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/ioh_logs/2D/std_0.0/f1/cmaes

=== Running DE on f1 (2D, noise=0.0, N=10 runs, Budget=1000000) ===
  Run  1/10: final clean error = 1.563194e-13
  Run  2/10: final clean error = 1.421085e-13
  Run  3/10: final clean error = 2.131628e-13
  Run  4/10: final clean error = 1.421085e-14
  Run  5/10: final clean error = 7.105427e-14
  Run  6/10: final clean error = 1.278977e-13
  Run  7/10